# DeepAgents 03 · 人工审核、记忆、子智能体、Skills

这一课把 DeepAgents 里四块「工程化能力」串成一条线，再加上一块官方补充：
**危险操作要人点头（人工审核）→ 跨会话记住用户（记忆）→ 复杂任务拆出去（子智能体）→
能力按需加载（Skills）→ 造自己的后端（官方补充）**。

全节五个概念，后面会反复出现：

| 概念 | 是什么 | 本节的代码形态 |
|---|---|---|
| 人工审核 HITL | 危险工具执行前挂起，等人批准/拒绝 | `interrupt_on={"write_file": True}` |
| 记忆 Memory | 记忆=文件，`memory=[...]` + `store=` 持久化 | `CompositeBackend` + `StoreBackend` |
| 子智能体 SubAgent | 主 Agent 通过 `task` 工具委派子任务 | `subagents=[SubAgent(...)]` |
| 技能 Skills | 一个文件夹 + 一个 `SKILL.md`，按需加载 | `skills=[父目录]` |
| 自定义后端 Backend | 继承 `BackendProtocol`，重写读写方法 | `class DictBackend(BackendProtocol)` |

> **本 notebook 由 `Agent/03_deepagents/` 下 9 个脚本合并而成**：
> `10_人工审核.py`（课案原版）+ `10_人工审核_jxsd.py`（完整版）、
> `11_记忆.py` + `11_记忆_jxsd.py`、
> `12_子智能体.py` + `12_子智能体_jxsd.py`、
> `13_skills.py` + `13_skills_jxsd.py`、
> `15_自定义后端_官方补充.py`（官方文档 backends.mdx 补缺）。

**官方文档**
- 定制（customization，含 interrupt_on / memory / subagents / skills）：<https://docs.langchain.com/oss/python/deepagents/customization>
- 子智能体（subagents）：<https://docs.langchain.com/oss/python/deepagents/subagents>
- 技能（skills）：<https://docs.langchain.com/oss/python/deepagents/skills>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 会真实调用 `.env` 里配置的大模型 |
| 依赖 | `deepagents` / `langgraph` / `langchain`（venv 已装） |
| 密钥 | `settings.api_key` / `base_url` / `model_name`（已配置） |
| 前置服务 | 「记忆」「云技能」两节额外需要**本机 PostgreSQL**（`settings.pg_uri`，已配置；未配置时这两节自动打印 `[跳过]`） |
| 预计耗时 | 约 2~5 分钟（多轮真实模型调用） |

> 本 notebook 里只有第 5 节「自定义后端」是**离线可复现**的（0 次模型调用）；
> 其余各节都要真实调模型，输出里的模型措辞每次都会不同，各节「预期输出」都做了标注。

## 本节地图

先看这一课五个主题的关系，以及「人工审核」一次中断的数据流。

```mermaid
graph LR
    A["人工审核<br/>interrupt_on 挂起"] --> B["记忆<br/>memory + Store"]
    B --> C["子智能体<br/>SubAgent 委派"]
    C --> D["Skills<br/>渐进式披露"]
    D --> E["自定义 Backend<br/>官方补充"]
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 主题 | 解决什么问题 | 关键参数 | 依赖 |
|---|---|---|---|
| 人工审核 | 危险工具执行前要人点头 | `interrupt_on` + `checkpointer` | 无（内存 checkpointer） |
| 记忆 | 跨会话记住用户偏好 | `memory` + `store` + `CompositeBackend` | PostgreSQL |
| 子智能体 | 上下文隔离、职责单一 | `subagents=[SubAgent(...)]` | 无 |
| Skills | 能力一次配置、按需加载 | `skills=[父目录]` | 本地磁盘 或 Store |
| 自定义后端 | 文件不在磁盘 / 要审计 / 要限流 | 继承 `BackendProtocol` | 无 |

## 0. 环境引导

notebook 的工作目录默认是它自己所在的文件夹，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。所以每个 notebook 的第一格
统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。

少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

In [ ]:
from config import settings

print("模型：", settings.model_name, " @ ", settings.base_url)
print("api_key 已配置：", bool(settings.api_key))
print("pg_uri 已配置：", bool(settings.pg_uri))

## 1. 人工审核（Human-in-the-Loop）

Agent 拿到 `write_file` / `execute` 这类危险工具之后，谁来兜底？答案不是靠提示词劝它
别乱来，而是把「执行前必须有人点头」做成**图的执行流程本身**：
`create_deep_agent(..., interrupt_on={"write_file": True})` 会在中间件栈尾部挂一个
`HumanInTheLoopMiddleware`，调用 `write_file` **之前**先挂起。

一次「暂停 → 审批 → 恢复」的完整链路：

1. 模型产出 tool_call（比如 `write_file` 写 hello.txt）；
2. HITL 中间件在工具真正执行前调用 LangGraph 的 `interrupt()`，图在这一步**冻结**成一份检查点；
3. `agent.invoke(...)` 立刻返回，结果里多出一个 `__interrupt__` 字段，装着待审批的请求：

   ```text
   {"action_requests": [{"name": 工具名, "args": 参数字典, "description": 说明}, ...],
    "review_configs": [...]}
   ```

4. 人类给出**与 action_requests 等长**的决策列表：

   ```text
   {"type": "approve"}                       批准
   {"type": "edit", "args": {...}}           改参数后执行
   {"type": "reject", "message": "拒绝理由"}  拒绝（理由会回传给模型）
   ```

5. 用 `Command(resume={"decisions": [...]})` 把决策喂回去，图从断点继续跑。

**两个必须记住的前提**：

- **必须配 checkpointer**：中断 = 图的执行状态被冻结，这份状态要有地方存，不传直接报错；
- 恢复必须用**同一个 `config`（同一个 thread_id）**，否则接不上原来那条执行线。

`interrupt_on` 有两种写法：`{"write_file": True}` 是简写（等价于允许三种决策），
`{"send_email": {"allowed_decisions": [...], "description": "..."}}` 是完整写法
（逐工具限制决策类型 + 补充说明）。

### 1.1 课案原版：最短实现

原版只做一件事：盯住 `write_file`，触发中断后展示待审批调用、读人工决定、返回 decisions。
它没有 `isatty()` 判断，`input()` 直接读控制台 —— 无头执行时 `input()` 会出问题，
所以下面用「预设答案函数」`scripted_input` 临时替换 `builtins.input`（照抄
`02_langchain/09_人工审核_jxsd.py` 的做法），演示「拒绝」这条路径。

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


@tool
def send_email(to: str, subject: str) -> str:
    """发送邮件（危险操作，需要人工批准）"""
    return f"已发送邮件给 {to}，主题：{subject}"

In [ ]:
# 课案原版只盯 write_file；send_email 在这里定义了但没挂上（挂上去要写 interrupt_on 完整写法，
# 见 1.2 之前的说明）。backend 用 FilesystemBackend(root_dir=".")，
# 但中断发生在写文件**之前**，被拒绝时磁盘上不会留下任何痕迹。
agent = create_deep_agent(
    model=llm,
    checkpointer=MemorySaver(),
    interrupt_on={"write_file": True},
    backend=FilesystemBackend(root_dir=".", virtual_mode=True),
)

config = {"configurable": {"thread_id": "1"}}

In [ ]:
def review_interrupts(hitl_request: dict) -> list[dict]:
    """展示待审批的工具调用，读取人工决定并返回 decisions 列表。

    :param hitl_request: HumanInTheLoopMiddleware 中断携带的审批请求，
        结构为 {"action_requests": [...], "review_configs": [...]}。
    :return: 与 action_requests 等长的决策列表，每项为 approve 或 reject。
    """
    action_requests = hitl_request["action_requests"]
    for action in action_requests:
        print(f"待审批工具: {action['name']}")
        print(f"工具参数: {action['args']}")
        if action.get("description"):
            print(f"说明: {action['description']}")

    # 非交互环境（stdin 被重定向 / 后台运行）下 input() 会直接抛 EOFError。
    # 审批是「有副作用的关卡」，读不到人就不放行 —— 默认拒绝并打印中文提示，
    # 而不是让整个脚本 traceback。要手工审批请在真实终端里运行本文件。
    try:
        choice = input("是否批准执行上述操作？[y/n]: ").strip().lower()
    except EOFError:
        print("\n[非交互环境] 读不到控制台输入，无人审批 → 默认拒绝执行。")
        print("             想手工审批请在真实终端里运行本文件。")
        return [{"type": "reject", "message": "非交互环境，未获得人工批准"}] * len(action_requests)

    if choice == "y":
        decision = {"type": "approve"}
    else:
        try:
            reason = input("拒绝原因（可选，直接回车跳过）: ").strip()
        except EOFError:
            reason = ""
        decision = {"type": "reject", "message": reason or "人工审核拒绝"}
    return [decision] * len(action_requests)

In [ ]:
def main() -> None:
    """运行代理；若触发审批中断则等待人工输入，否则直接输出结果。"""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "写一个hello.txt"}]}, config
    )

    interrupts = result.get("__interrupt__")
    if not interrupts:
        # 模型未调用 write_file，无需审批
        print("未触发审批，最终结果：", result["messages"][-1].content)
        return

    # deepagents 0.7.x 中 state.next 为 ('HumanInTheLoopMiddleware.after_model',)
    state = agent.get_state(config)
    print(f"审批中断点: {state.next}")

    decisions = review_interrupts(interrupts[0].value)
    result = agent.invoke(Command(resume={"decisions": decisions}), config)
    print(result["messages"][-1].content)


# 预设答案函数：把 builtins.input 临时换成「按顺序吐答案」，用完按 default 兜底。
# 这样跑的还是课案原样的 review_interrupts()（同一条代码路径），而不是另写假逻辑。
DEFAULT_DECISION = "approve"


def scripted_input(answers: list[str], default: str = DEFAULT_DECISION):
    """把 builtins.input 临时换成「按顺序吐预设答案」，用于无头执行。

    预设答案用完后按 default 兜底，并在日志里说明 —— 让「脚本喂的答案不够」
    这件事可见，而不是静默改变演示语义。
    """
    answers = list(answers)

    def fake_input(prompt: str = "") -> str:
        if answers:
            answer = answers.pop(0)
        else:
            answer = default
            print(f"  （预设答案已用完，按兜底值处理：{answer}）")
        print(f"{prompt}{answer}")
        return answer

    return fake_input


import builtins

_orig_input = builtins.input
# 喂两个答案：第一个 "n"（拒绝），第二个 ""（拒绝原因留空 → 落到默认文案）。
# 想真实交互，把下面这一行和 finally 里的还原去掉，自己敲 y / n。
builtins.input = scripted_input(["n", ""])
try:
    main()
finally:
    builtins.input = _orig_input

### 预期输出

```text
审批中断点: ('HumanInTheLoopMiddleware.after_model',)
待审批工具: write_file
工具参数: {'file_path': 'hello.txt', ...}
是否批准执行上述操作？[y/n]: n
拒绝原因（可选，直接回车跳过）:
（模型收到「人工审核拒绝」后的回应，措辞每次不同）
```

> ⚠️ 上面模型最后的回应**由模型决定、每次不同**；只有「审批中断点 / 待审批工具 / 工具参数」
> 这些结构是固定的。**若模型这一轮没发起 write_file**，会打印 `未触发审批，最终结果：…`
> 并直接结束（本机模型偶发）。真实交互请去掉 `builtins.input = scripted_input(...)` 那两行。

### 1.2 完整版：落盘核对 + 自动批准

完整版相对课案的两处工程化改动：

1. `root_dir` 从课案的 `"."` 改成 `WORKDIR / "hitl_demo"`（不把 hello.txt 写到仓库根）；
2. `review_interrupts` 加了 `sys.stdin.isatty()` 判断：**交互时行为与课案完全一致**，
   非交互时打印提示并自动批准，保证无人值守也能跑完。

下面这一节演示「批准」路径：批准后 `write_file` 真正执行，磁盘上留下 hello.txt。

In [ ]:
from pathlib import Path

from deepagents.backends import FilesystemBackend  # 改用真实磁盘
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# 课案是 Path(__file__).resolve().parent / "tmp_jxsd_deepagents_hitl"；
# notebook 没有 __file__，且 WORKDIR 是同章共享容器，套一层专属子目录。
_NB_WORKDIR = WORKDIR               # 先记住共享容器，本节末尾还原，避免污染后面各节的 SKILLS_ROOT
WORKDIR = WORKDIR / "hitl_demo"

agent = create_deep_agent(
    model=llm,
    backend=FilesystemBackend(root_dir=str(WORKDIR), virtual_mode=True),  # 文件写到真实磁盘
    # 盯住 write_file：只要模型要写文件就挂起等审批。
    # True 是简写，等价于 allowed_decisions=["approve","edit","reject"]。
    interrupt_on={"write_file": True},
    # 中断状态要有地方存 —— 不配 checkpointer 直接报错
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "1"}}

In [ ]:
def review_interrupts(hitl_request: dict) -> list[dict]:
    """展示待审批的工具调用，读取人工决定并返回 decisions 列表。

    :param hitl_request: HumanInTheLoopMiddleware 中断携带的审批请求，
        结构为 {"action_requests": [...], "review_configs": [...]}。
    :return: 与 action_requests 等长的决策列表，每项为 approve 或 reject。
    """
    action_requests = hitl_request["action_requests"]
    for action in action_requests:
        print(f"待审批工具: {action['name']}")
        print(f"工具参数: {action['args']}")
        if action.get("description"):
            print(f"说明: {action['description']}")

    # ---------- 完整版新增：非交互终端自动批准 ----------
    # 课案这里是裸的 input()。在管道/重定向/CI 里 input() 会抛 EOFError 把脚本打断，
    # 所以加一层判断：不是交互终端就自动批准，并把「本该问什么」打印出来。
    if not sys.stdin.isatty():
        print("（非交互终端：跳过 input()，自动批准本次操作）")
        return [{"type": "approve"}] * len(action_requests)

    try:
        choice = input("是否批准执行上述操作？[y/n]: ").strip().lower()
    except EOFError:
        # 有些环境 isatty() 为真但 stdin 已关闭，这里兜一下，避免脚本崩掉
        print("（stdin 已关闭，自动批准本次操作）")
        return [{"type": "approve"}] * len(action_requests)

    # 批准 → approve；其他任何输入（含直接回车）都按拒绝处理，并且把理由回传给模型
    if choice == "y":
        decision = {"type": "approve"}
    else:
        try:
            reason = input("拒绝原因（可选，直接回车跳过）: ").strip()
        except EOFError:
            # 同一层兜底：拿不到理由就用空串，后面会给一个默认文案
            reason = ""
        # message 会作为工具反馈交回模型，所以即便用户没写理由也要给一句人话
        decision = {"type": "reject", "message": reason or "人工审核拒绝"}
    # 一次中断可能攒了多个待审批调用，用同一个决定铺满，长度必须与 action_requests 对齐
    return [decision] * len(action_requests)

In [ ]:
def main() -> None:
    """运行代理；若触发审批中断则等待人工输入，否则直接输出结果。"""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "写一个hello.txt"}]}, config
    )

    interrupts = result.get("__interrupt__")
    if not interrupts:
        # 模型未调用 write_file，无需审批
        print("未触发审批，最终结果：", result["messages"][-1].content)
        return

    # deepagents 0.7.x 中 state.next 为 ('HumanInTheLoopMiddleware.after_model',)
    # get_state 能看到「图停在哪一步」，是排查中断问题最直接的手段。
    state = agent.get_state(config)
    print(f"审批中断点: {state.next}")

    # interrupts[0].value 就是上面 review_interrupts 的入参；
    # 一次可能攒了多个待审批调用，所以 decisions 要按数量对齐。
    decisions = review_interrupts(interrupts[0].value)
    print(f"人工决策: {decisions}")

    # Command(resume=...) 把决策送回被冻结的图，从断点继续执行
    result = agent.invoke(Command(resume={"decisions": decisions}), config)
    print(result["messages"][-1].content)


WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"工作目录：{WORKDIR}\n")

# 同上：isatty() 为假时自动批准，无需 input；万一 isatty 判真，scripted_input 喂 "y" 兜底。
builtins.input = scripted_input(["y"])
try:
    main()
finally:
    builtins.input = _orig_input

print("\n===== 磁盘结果 =====")
files = sorted(p.name for p in WORKDIR.rglob("*") if p.is_file())
print(f"  {WORKDIR.name}/ 下的文件：{files if files else '（空）'}")
print(
    "\n要点回顾：\n"
    "  - 中断发生在工具真正执行**之前**，所以被拒绝时磁盘上不会留下任何痕迹；\n"
    "  - decisions 必须和 action_requests 等长，一次中断可能攒了多个调用；\n"
    "  - 拒绝时带上 message，模型会收到这条反馈并自行调整方案。"
)

WORKDIR = _NB_WORKDIR               # 还原共享容器，后面各节（SKILLS_ROOT 等）还要用它

### 预期输出

```text
工作目录：…\03_deepagents\tmp_nb_work\hitl_demo

审批中断点: ('HumanInTheLoopMiddleware.after_model',)
待审批工具: write_file
工具参数: {'file_path': 'hello.txt', ...}
（非交互终端：跳过 input()，自动批准本次操作）
人工决策: [{'type': 'approve'}]
（模型确认写入的回应，措辞每次不同）

===== 磁盘结果 =====
  hitl_demo/ 下的文件：['hello.txt']
```

> ⚠️ 模型最后一句回应**由模型决定、每次不同**；`工作目录` 里的绝对路径也**因机器而异**。
> 稳定的部分是：审批中断点、待审批工具 `write_file`、`人工决策: [{'type': 'approve'}]`、
> 以及磁盘上最终出现 `hello.txt`。**若模型这轮没发起 write_file**，则打印 `未触发审批`、磁盘为空。

## 2. 记忆（memory）

DeepAgents 的记忆**不走向量库检索**，而是**通过文件系统实现**：Agent 把记忆当文件读写，
backend 决定这些文件存在哪、谁能访问。`memory=[...]` 声明哪些路径算记忆，
`MemoryMiddleware` 把内容拼进**系统提示词**（所以 Agent 一开场就"记得"），
同时告诉模型「学到新东西要 update 这个文件」。

两种记忆的区别（课案总表）：

| 记忆类型 | 机制 | 生命周期 |
|---|---|---|
| 短期记忆 | checkpoint 自动保存 messages，同一 thread_id 内多轮对话共享 | 同 thread |
| 长期记忆 | `memory=["/memories/..."]` + StoreBackend，文件持久化到 PostgreSQL | 跨 thread、跨会话 |

对应到代码就是两个参数各管一摊：

- `checkpointer=checkpointer` → 短期记忆（对话历史）
- `store=store` + `memory=[...]` → 长期记忆（记忆文件）

**为什么要 CompositeBackend 而不是裸 StoreBackend**：只有 `/memories/` 需要持久化，
其余路径（框架内部的 `/large_tool_results/`、`/conversation_history/`）留在内存里更干净。

记忆作用域怎么选（课案表）：

| 作用域 | namespace | 谁共享 |
|---|---|---|
| Agent 级 | `(assistant_id,)` | 所有用户共享同一个 Agent 的记忆 |
| 用户级 | `(user.identity,)` | 每个用户独立 |
| 组织级 | `(org_id,)` | 整个组织共享 |

本地跑拿不到 `rt.server_info`（那是 LangGraph Server 平台才注入的），所以本地版用固定
namespace 替代。

### 2.1 课案原版（带一个真实 bug）

课案原版预存记忆时用的是 `/memories/AGENTS.md` 这个 key，但 `CompositeBackend` 的路由语义
是**把 `/memories/` 前缀剥掉再转给 StoreBackend**，StoreBackend 又**拿路径直接当 Store key**。
两边一凑：

- 预存的 key `/memories/AGENTS.md`，在 Agent 眼里出现在 `/memories/memories/AGENTS.md`
- Agent 按提示词去 `read_file("/memories/AGENTS.md")` → `File '/AGENTS.md' not found`

下面这一节原样跑课案原版，**亲眼看到这个 bug**（完整版 2.2 才修）。

In [ ]:
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore

from deepagents.backends import StateBackend, StoreBackend, CompositeBackend

In [ ]:
with (
    PostgresStore.from_conn_string(settings.pg_uri) as store,
    PostgresSaver.from_conn_string(settings.pg_uri) as checkpointer,
):
    store.setup()
    checkpointer.setup()

    # namespace 是可调用对象：根据运行时上下文动态生成命名空间。
    # 这里演示用固定命名空间，实际可按 user_id / thread_id 划分。
    backend = StoreBackend(store=store, namespace=lambda _rt: ("user-1001", "filesystem"))

    agent = create_deep_agent(
        model=llm,
        backend=backend,
        system_prompt="你是文档助手，可以把重要文档写入文件系统。",
    )

    existing = store.get(("my-agent",), "/memories/AGENTS.md")
    if existing is None:
        store.put(
            ("my-agent",),
            "/memories/AGENTS.md",
            create_file_data("""回复风格 回复简洁，不超过三句话"""),
        )
        print("初始化默认记忆文件")
    else:
        print("记忆文件已存在，保留现有内容")

    agent = create_deep_agent(
        model=llm,
        memory=["/memories/AGENTS.md"],
        system_prompt="你的记忆文件在 /memories/AGENTS.md。每次回复前先 read_file 读取记忆，学到新信息后用 edit_file 更新该文件。",
        backend=CompositeBackend(
            default=StateBackend(),
            routes={
                "/memories/": StoreBackend(
                    namespace=lambda rt: ("my-agent",),
                ),

            },
        ),
        store=store,  # 长期记忆
        checkpointer=checkpointer,  # 短期记忆（对话历史）
    )

    # Thread 1：Agent 学到新偏好，自动写入记忆
    config1 = {"configurable": {"thread_id": "1"}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "我叫小明，我喜欢长篇大论"}]},
        config=config1,
    )
    for m in result["messages"]:
        print(f"m.content:{m.content}")
        print(f"m.name:{m.name}")
        print("=" * 50)

    # 验证是否修改了
    mem = store.get(("my-agent",), "/memories/AGENTS.md")
    print(mem.value['content'])
    print("----" * 50)

    result = agent.invoke(
        {"messages": [{"role": "user", "content": "先把我刚刚说的话重复一边，写一篇咖啡店小红书帖子"}]},
        config=config1,
    )

    print(result["messages"][-1].content)

    print("----" * 50)

    # Thread 2：Agent 读取记忆，应用之前的偏好
    config2 = {"configurable": {"thread_id": "2"}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "先把我刚刚说的话重复一边，写一篇咖啡店小红书帖子"}]},
        config=config2,
    )
    print(result["messages"][-1].content)

### 预期输出

```text
记忆文件已存在，保留现有内容
m.content:我叫小明，我喜欢长篇大论
m.name:None
==================================================
...
m.content:Error: File '/AGENTS.md' not found
m.name:read_file
...
m.content:['/memories/memories/']
m.name:ls
...
回复风格 回复简洁，不超过三句话
--------------------------------------------------
...
```

> ⚠️ 这一格**几乎所有正文都由模型决定**（它读文件、报错、重试、写帖子的每一步措辞都不同），
> 上面只是本机一次实测的片段，**别逐字比对**。真正稳定的是两处：`File '/AGENTS.md' not found`
> （预存 key 多了一层 `/memories/`）和 `ls` 返回 `['/memories/memories/']` —— 这正是 2.2 要修的 bug。
> 另外课案原版用固定 `thread_id="1"`，会和同一张 PG checkpoint 表里其它脚本的 `"1"` 串历史
> （实测一开场会冒出上一份代码留下的消息）。

### 2.2 完整版：修掉双重前缀 bug

完整版的修法：**Store key 要写成「剥掉路由前缀之后」的路径** —— 预存记忆用 `/AGENTS.md`
而不是 `/memories/AGENTS.md`。另外：

- 记忆命名空间换成 `("jxsd-my-agent",)`，thread_id 换成带前缀的 `jxsd-mem-*`，
  避免和 2.1 原版（`("my-agent",)` + `"1"`）写同一个 PostgreSQL 互相覆盖；
- `RESET_MEMORY_ON_START = True` 每次跑先清空本命名空间，让主线每次都完整走一遍；
- system_prompt 补了「工具用法提醒」：`read_file` 返回**每行带行号**，`edit_file` 的
  `old_string` 必须填**去掉行号后的原文**；报 String not found 就别重试，直接 `write_file` 写全文。

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore

# 课案这里是写死的连接串；本项目一律走配置（口令只存在于 .env 里）
DB = settings.pg_uri

# 记忆文件的路径（Agent 视角的路径），也是交给 MemoryMiddleware 的 sources
MEMORY_PATH = "/memories/AGENTS.md"

# Agent 访问 /memories/AGENTS.md 时，CompositeBackend 会剥掉 /memories/ 前缀，
# StoreBackend 实际拿到的路径（= Store 里的 key）就是 /AGENTS.md。
# 预存记忆必须用这个 key，否则 Agent 读不到（详见 2.1 那个 bug）。
MEMORY_KEY = "/AGENTS.md"

# 记忆的命名空间。课案本地版写死 ("my-agent",)；生产环境改成
# lambda rt: (rt.server_info.user.identity,) 就能做到「每个用户一份记忆」。
MEMORY_NAMESPACE = ("jxsd-my-agent",)

# 演示开关：为 True 时每次运行先清空本文件自己的记忆，让
# 「初始化默认记忆 → Agent 学到新偏好 → 改写记忆 → 新 thread 读记忆」这条主线
# 每次都完整走一遍（不清空的话，第二次跑只会打印「记忆文件已存在，保留现有内容」）。
RESET_MEMORY_ON_START = True

In [ ]:
if not DB:
    print("[跳过] 未配置 settings.pg_uri（.env 里的 PG_URI），无法演示长期记忆。")
    sys.exit(0)

# 两个上下文管理器一起开：一个 Store（长期记忆），一个 Checkpointer（短期记忆）。
# PostgresStore / PostgresSaver 都是连接池封装，用 with 保证连接被释放。
with (
    PostgresStore.from_conn_string(DB) as store,
    PostgresSaver.from_conn_string(DB) as checkpointer,
):
    # 首次使用前必须建表：store 建 store 相关表，checkpointer 建 checkpoint 相关表。
    # 重复调用是幂等的，所以每次跑都调一遍没问题。
    store.setup()
    checkpointer.setup()

    # ---------- 1. 预存记忆（长期记忆的初始内容） ----------
    # 演示用：先清空本文件命名空间下的旧记忆，保证每次跑都是同一个起点。
    if RESET_MEMORY_ON_START:
        for item in store.search(MEMORY_NAMESPACE):
            store.delete(MEMORY_NAMESPACE, item.key)
        print(f"已清空 namespace={MEMORY_NAMESPACE} 下的历史记忆（演示用）")

    # 先查一次，避免每次都把用户已经改过的记忆覆盖回默认值。
    existing = store.get(MEMORY_NAMESPACE, MEMORY_KEY)
    if existing is None:
        store.put(
            MEMORY_NAMESPACE,
            MEMORY_KEY,
            # create_file_data 生成的是 backend 认识的 FileData 结构
            # （content / encoding / created_at / modified_at），
            # 直接 put 一个裸字符串 backend 读不出来。
            create_file_data("""回复风格 回复简洁，不超过三句话"""),
        )
        print("初始化默认记忆文件")
    else:
        print("记忆文件已存在，保留现有内容")

    # 打印一下 Store 里真实的 key，方便把「路径」和「Store key」对上号
    print(f"Store namespace={MEMORY_NAMESPACE} 里的 key：{[i.key for i in store.search(MEMORY_NAMESPACE)]}")

    # 记下「更新前」的记忆内容，跑完 Thread 1 后好做对比
    _before = store.get(MEMORY_NAMESPACE, MEMORY_KEY)
    before_text = _before.value["content"] if _before else ""

    agent = create_deep_agent(
        model=llm,
        # 声明哪些文件算「记忆」：MemoryMiddleware 会把它们注入系统提示词
        memory=[MEMORY_PATH],
        # 课案原文的 system_prompt 是：
        #   "你的记忆文件在 /memories/AGENTS.md。每次回复前先 read_file 读取记忆，
        #    学到新信息后用 edit_file 更新该文件。"
        # 这里在原文后面补了两句「工具用法」提示，原因是实测踩到的坑：
        #   read_file 的返回**每行前面带行号**（形如 `1  正文`），
        #   模型会把带行号的整行当成 old_string 丢给 edit_file，
        #   而磁盘/Store 里的真实内容是没有行号的 →
        #   于是反复报 "String not found in file"，记忆一直改不动。
        # 把格式说清楚、并给一条 write_file 兜底路径，更新就能成功。
        system_prompt=(
            "你的记忆文件在 /memories/AGENTS.md。每次回复前先 read_file 读取记忆，"
            "学到新信息后用 edit_file 更新该文件。\n"
            "工具用法提醒：read_file 返回的每一行前面有形如 `1  ` 的行号，"
            "edit_file 的 old_string 必须填**去掉行号后的原文**；"
            "edit_file 一旦报 String not found，就别再重试，"
            "直接用 write_file 把整理好的**新记忆全文（不带行号）**写进去 —— 这样最稳。"
        ),
        backend=CompositeBackend(
            # 兜底留在内存，只有 /memories/ 走持久化
            default=StateBackend(),
            routes={
                "/memories/": StoreBackend(
                    # 生产改成 lambda rt: (rt.server_info.user.identity,)
                    namespace=lambda rt: MEMORY_NAMESPACE,
                ),
            },
        ),
        store=store,                # 长期记忆（记忆文件的物理存储）
        checkpointer=checkpointer,  # 短期记忆（对话历史）
    )

    # ---------- 2. Thread 1：Agent 学到新偏好，自动写入记忆 ----------
    config1 = {"configurable": {"thread_id": "jxsd-mem-1"}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "我叫xx，我喜欢长篇大论"}]},
        config={**config1, "recursion_limit": 60},
    )
    # 把**完整消息轨迹**打出来（课案也是这么做的）：
    # 这里能直接看到 Agent 到底有没有 read_file / edit_file / write_file 记忆文件，
    # 比只看最后一条回答信息量大得多。
    for m in result["messages"]:
        print(f"m.content:{m.content}")
        print(f"m.name:{m.name}")
        print("=" * 50)

    # 验证是否修改了
    # 注意这里绕开 Agent 直接读 Store —— 这是「记忆真的落库了」的硬证据
    mem = store.get(MEMORY_NAMESPACE, MEMORY_KEY)
    after_text = mem.value["content"] if mem else ""
    print(after_text)
    print(f"记忆是否被更新：{'是' if after_text != before_text else '否（本轮模型没有改写记忆文件）'}")
    print("----" * 50)

    # 同一 thread 的第二问：这次要的是「应用刚学到的偏好」。
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "先把我刚刚说的话重复一边，写一篇咖啡店小红书帖子"}]},
        config={**config1, "recursion_limit": 60},
    )

    print(result["messages"][-1].content)

    print("----" * 50)

    # ---------- 3. Thread 2：**新会话**读取记忆，应用之前的偏好 ----------
    # 这里是最关键的一步：thread_id 换了，短期记忆（对话历史）是空的，
    # 但 Agent 一上来仍然知道「用户喜欢长篇大论」—— 因为记忆走的是 Store，
    # 由 MemoryMiddleware 注入系统提示词，跟 thread 无关。
    config2 = {"configurable": {"thread_id": "jxsd-mem-2"}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "先把我刚刚说的话重复一边，写一篇咖啡店小红书帖子"}]},
        config={**config2, "recursion_limit": 60},
    )
    print(result["messages"][-1].content)

    print("\n" + "=" * 60)
    # 下面这一段是「看得见的部分」——只陈述**机制**上必然成立的事，
    # 不保证模型一定照做（模型服从度有波动，见下面的实测观察）
    print("结论（看得见的部分）：")
    print(f"  1) 记忆文件的物理落点：PostgresStore 的 namespace={MEMORY_NAMESPACE}，key='{MEMORY_KEY}'")
    print(f"  2) 记忆文件当前内容：{after_text!r}")
    print("  3) thread 2 是**全新会话**（thread_id 不同，对话历史为空），")
    print("     但 MemoryMiddleware 会把上面这份记忆文件注入它的系统提示词 ——")
    print("     所以「跨会话记住用户偏好」这件事在机制上是成立的，跟 thread_id 无关。")
    print()
    # 「实测观察」段：把不确定性明说，避免学生把模型的行为波动当成代码 bug
    print("实测观察（务必自己跑一遍看）：")
    print("  - 模型是否**真的按记忆行事**、是否**真的改写记忆文件**，取决于模型服从度，会有波动；")
    print("    本文件用 grok-4.6 实测：有时能成功改写记忆（输出里出现")
    print("    Successfully replaced 1 instance(s)），有时它会连着几次 edit_file 失败就放弃。")
    print("  - 最常见的失败原因是 read_file 返回的内容**带行号**，模型把带行号的整行"
          "当成 old_string 传给 edit_file；")
    print("    上面的 system_prompt 已经专门提醒过这一点，但提示词不是强约束。")
    print("  - 想干净复现，把 RESET_MEMORY_ON_START 置 True 再跑（默认就是 True）。")

### 预期输出

```text
已清空 namespace=('jxsd-my-agent',) 下的历史记忆（演示用）
初始化默认记忆文件
Store namespace=('jxsd-my-agent',) 里的 key：['/AGENTS.md']
m.content:我叫xx，我喜欢长篇大论
...
记忆是否被更新：是（或 否，取决于模型这轮是否改写）
...
结论（看得见的部分）：
  1) 记忆文件的物理落点：PostgresStore 的 namespace=('jxsd-my-agent',)，key='/AGENTS.md'
  2) 记忆文件当前内容：…
```

> ⚠️ 这一格**模型正文（消息轨迹、帖子正文、是否真的改写记忆）每次运行都不同**，别逐字比对。
> 稳定的是：`已清空 namespace=('jxsd-my-agent',)`、`Store ... 里的 key：['/AGENTS.md']`
> 以及最后「结论（看得见的部分）」那几行。

## 3. 子智能体（SubAgent）

主 Agent 可以通过 `task` 工具把子任务委派给子 Agent。每个子 Agent 有独立的系统提示词、
独立的工具集、**独立上下文窗口**。它解决的是一个 Agent 干所有活的三个代价：

1. **上下文隔离**：子 Agent 翻十几篇资料产生的中间信息不塞进主 Agent 上下文，只回传结论；
2. **职责单一**：每个子 Agent 只擅长一件事，提示词可以写得很窄；
3. **模型/工具可不同**：调研用便宜模型、写代码用强模型，按需分配。

`SubAgent` 的字段（`deepagents.middleware.subagents.SubAgent`，一个 TypedDict）：

| 字段 | 必需 | 说明 |
|---|---|---|
| name | 是 | 唯一标识。主 Agent 调 task 时用它选人 —— **不能重名** |
| description | 是 | 干什么用的。主 Agent 靠这句话决定「这个活该不该外包出去」 |
| system_prompt | 否 | 子 Agent 的人设与工作规范 |
| tools | 否 | 子 Agent 专属工具集；不填则继承主 Agent 的工具 |
| model | 否 | 覆盖主 Agent 的模型，格式 'provider:model-name' |
| middleware | 否 | 追加中间件（限流、日志等） |
| interrupt_on | 否 | 给这个子 Agent 单独配人工审核（需要 checkpointer） |
| skills | 否 | 这个子 Agent 能用的技能目录 |

`SubAgent` 是 TypedDict，所以 `SubAgent(name=..., description=...)` 和直接写
`{"name": ..., "description": ...}` 完全等价。

### 3.1 课案原版：一个 researcher

原版只配了一个 `researcher` 子 Agent，并给它一个 `fake_search` 搜索工具（模拟结果，不联网）。

In [ ]:
from langchain_core.tools import tool


@tool
def fake_search(keyword: str) -> str:
    """搜索资料。keyword：关键词"""
    return f"「{keyword}」相关资料：LangGraph……（模拟搜索结果）"


research_agent = create_deep_agent(
    model=llm,
    tools=[fake_search],
    system_prompt="你是总编，调研任务委派给 researcher 子智能体，然后汇总成一句话结论。",
    subagents=[
        {
            "name": "researcher",
            "description": "资料调研专家，擅长搜索和整理资料",  # 给主 Agent 看的「外包说明」
            "system_prompt": "你是调研专家，用搜索工具收集资料，输出要点列表。",
            "tools": [fake_search],   # 子 Agent 独立工具集
            "model": llm,             # 子 Agent 可指定不同模型（传实例最稳妥）
        },
        # 可以继续添加更多子智能体，如 coder / writer / reviewer …
    ],
)

In [ ]:
result = research_agent.invoke(
    {"messages": [("user", "调研一下 LangGraph 并给我一个简短结论")]},
    config={"recursion_limit": 80},
)
print("AI：", result["messages"][-1].content)

### 预期输出

```text
AI：LangGraph 是一个…（模型给出的简短结论，措辞每次不同）
```

> ⚠️ 结论正文**由模型决定、每次不同**；只有前缀 `AI：` 是固定的。

### 3.2 完整版：三个不重名的子智能体 + 内省 task 工具

**课案原文的一个坑**：它写了三个 `SubAgent(name="researcher", ...)` —— 名字一模一样，明显是
复制粘贴没改。实测这样写会直接报错：

```text
ValueError: Duplicate subagent name 'researcher'; each subagent must have a unique name.
```

所以完整版改成三个**不同职责**的子 Agent（researcher / writer / reviewer）。
配了 subagents 之后，主 Agent 的工具清单里会多出一个 `task`，下面用内省把它打出来。

In [ ]:
from deepagents import SubAgent, create_deep_agent

# SubAgent：声明式子 Agent，主 Agent 通过 task 工具自动委派
# 注意三个子 Agent 的**名字各不相同**（researcher / writer / reviewer）——
# 课案原文三个都叫 researcher，实测会直接抛
# ValueError: Duplicate subagent name（详见 3.2 之前的说明）。
# description 是写给**主 Agent 的模型**看的：它靠这句话决定「这个活该不该外包给谁」，
# 所以写得越具体，委派越准（本文件三条 description 分别对应调研/写作/审查三种活）。
agent = create_deep_agent(
    model=llm,
    system_prompt="你是项目经理，复杂任务委派给子Agent执行。",
    subagents=[
        SubAgent(
            name="researcher",
            description="深度调研一个课题，返回结构化报告",
            # 让子 Agent 在自己的回答里留个记号，方便在消息轨迹里认出
            # 「这段是子 Agent 产出的」——课案用 '[调研中]' 就是这个目的。
            system_prompt="你是调研专家。回应的第一句加上'[调研中]'。",
        ),
        SubAgent(
            name="writer",
            description="把调研要点改写成通俗易懂的科普短文",
            # 每个子 Agent 有独立上下文：它只看得到 task 传进去的那段任务描述，
            # 看不到主 Agent 的整段对话 —— 这正是「上下文隔离」的落地方式。
            system_prompt="你是科普作者。回应的第一句加上'[写作中]'。输出不超过 200 字。",
        ),
        SubAgent(
            name="reviewer",
            description="审查文案的事实准确性与逻辑漏洞，指出问题并给出修改建议",
            # 给子 Agent 留「可辨识的记号」（[审稿中]）纯粹是为了教学观察：
            # 在 ToolMessage 里一眼认出这段产出是哪个子 Agent 写的
            system_prompt="你是严格的审稿人。回应的第一句加上'[审稿中]'。只挑毛病，不要重写全文。",
        ),
    ],
)


def _list_tools(agent_) -> list[str]:
    """内省：列出主 Agent 挂载的工具名（配了 subagents 之后应该能看到 task）。"""
    try:
        return sorted(agent_.nodes["tools"].bound.tools_by_name)
    except Exception:                      # noqa: BLE001 —— 内省失败不影响主流程
        return []


print("主 Agent 的工具清单（注意 task）：")
for name in _list_tools(agent):
    print(f"  - {name}")
print()

result = agent.invoke(
    {"messages": [{"role": "user", "content": "调研一下Python异步编程"}]},
    # 子 Agent 会再各跑一轮，步数需求比单 Agent 大，放宽上限
    config={"recursion_limit": 80},
)

# 课案是 for m in result["messages"]: print(m) —— 把**完整消息轨迹**打出来。
# 这里面藏着子 Agent 的产出（以 ToolMessage 的形式回到主 Agent），
# 比只看最后一条结论信息量大得多。
print("=" * 60)
print("完整消息轨迹：")
print("=" * 60)
for m in result["messages"]:
    print(m)
    print("-" * 60)

print("\n最终回答：")
print(result["messages"][-1].content)

### 预期输出

```text
主 Agent 的工具清单（注意 task）：
  - task
  - ...（其它内置工具）

============================================================
完整消息轨迹：
============================================================
...（含 [调研中] / [写作中] / [审稿中] 等子 Agent 产出的 ToolMessage）

最终回答：
（项目经理汇总后的结论，措辞每次不同）
```

> ⚠️ 工具清单的具体条目、消息轨迹、最终回答**都由框架和模型决定、每次运行不同**；
> 稳定的结构是「工具清单里出现 `task`」和「消息轨迹里出现子 Agent 的记号（[调研中] 等）」。

## 4. Skills（技能）：按需加载的「操作手册」

Skills 把智能体的能力封装成「一个文件夹 + 一个 `SKILL.md`」，一次配置、永久复用，
而不是每次往提示词里粘一大段。它**为什么省 Token**：三步「渐进式披露」——

1. **选择**：启动时只读每个技能的 `name` + `description`（几十个字符）
2. **学习**：任务匹配到某个技能，才把该 SKILL.md 的正文加载进来
3. **使用**：按指示执行，可参考其他文件（references/）或跑 scripts/

Skills 和传统 Prompt 的区别（课案三条）：

| 维度 | 普通 Prompt | Skills |
|---|---|---|
| 配置成本 | 每次使用都要重新配置 | 一次配置，永久复用 |
| Token 效率 | 每次调用全量加载 | 按需懒加载，只加载对应内容 |
| 维护成本 | 跨场景反复复制粘贴 | 只改 SKILL.md，全局/项目级统一生效 |

目录结构（一个文件夹 = 一个技能）：

```text
my-skill/
├── SKILL.md      # 必需：指令 + 元数据（YAML frontmatter）
├── scripts/      # 可选：执行脚本
├── references/   # 可选：文档资料（渐进式披露，用到才读）
└── assets/       # 可选：资源
```

`SKILL.md` 的 frontmatter 必填 `name`（≤64 字符，小写字母/数字/`-`）和 `description`
（≤1024 字符）。**注意 `skills` 传的是技能的【父目录】，不是 SKILL.md 本身** —— 这是本节
最容易搞错的一点。

### 4.1 课案原版：现场造一个技能

原版现场 `write_text` 造一个 `report-writer` 技能目录，然后 `skills=[str(skill_dir)]` 挂上。
课案原文的 `skill_dir = Path("tmp_skill_report_writer")` 会落到仓库根，这里改成
`WORKDIR / "skills_demo"`，避免污染仓库。

In [ ]:
from pathlib import Path

# ---------- 1. 动态创建一个技能目录（实际项目中提前准备好） ----------
skill_dir = WORKDIR / "skills_demo"
skill_dir.mkdir(exist_ok=True)
(skill_dir / "SKILL.md").write_text(
    """---
name: report-writer
description: 撰写正式工作周报时使用本技能
---

# 周报撰写规范

1. 结构固定为三段：本周完成 / 下周计划 / 风险与求助
2. 每条用「动宾短语」开头，例如「完成了 XX 模块开发」
3. 总字数控制在 200 字以内
""",
    encoding="utf-8",
)

# ---------- 2. 创建带技能的 Deep Agent ----------
agent = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt="你是职场写作助手。",
    skills=[str(skill_dir)],  # 传入技能目录，Agent 会按需加载
)

result = agent.invoke(
    {"messages": [("user", "帮我写本周周报：完成了登录模块，下周做支付")]},
    config={"recursion_limit": 50},
)
print("AI：", result["messages"][-1].content)

### 预期输出

```text
AI：（按 report-writer 技能的三段式结构写出的周报，正文措辞每次不同）
```

> ⚠️ 周报正文**由模型决定、每次不同**；稳定的只有「它应当遵循 SKILL.md 里
> 三段式 + 动宾短语 + 200 字以内的规范」。想看到「是否真的 read_file 拉了 SKILL.md」，
> 看 4.2 的 updates 轨迹。

### 4.2 完整版：本地技能（渐进披露证据）+ 云技能（Store 隔离）

完整版分两部分：

- **Part 1 本地技能**：用 `LocalShellBackend`（技能里的脚本要能执行），
  并用 `stream_mode="updates"` 把工具调用轨迹打出来 —— 这是「渐进式披露」最直接的证据：
  启动时系统提示里只有 description，模型决定用技能之后才会 `read_file` 拉 SKILL.md 正文。
- **Part 2 云技能**：技能仓库也可以不是磁盘而是 Store（`skills=["/skills/"]` +
  `CompositeBackend` 路由到 StoreBackend），靠 namespace 做多用户隔离。

**云技能为什么必须挂在 CompositeBackend 上**：`skills=["/skills/"]` 只是个路径前缀，
真正决定「去哪读」的是 backend。而且预存技能时 Store key 要写成「剥掉 `/skills/` 前缀」
之后的路径（`/code-review/SKILL.md`），否则 `ls("/skills/")` 会得到 `/skills/skills/`，
技能发现数 = 0（静默失效，不报错）。这和 2.1 记忆那个坑是**同一个坑**。

In [ ]:
import os
from pathlib import Path

from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from deepagents.backends.local_shell import LocalShellBackend
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore

# 课案是 Path(__file__).resolve().parent / "tmp_jxsd_deepagents_skills"；
# notebook 没有 __file__，改到 WORKDIR 下的专属子目录（同章共享容器，必须套一层）。
SKILLS_ROOT = WORKDIR / "deepagents_skills"

# 和 06 文件同样的原因：把当前 venv 的 Scripts 目录顶到 PATH 最前面，
# 免得技能里的脚本被 PATH 中那个「假的 python」静默吃掉。
VENV_SCRIPTS = str(Path(sys.executable).resolve().parent)


def build_local_skill() -> Path:
    """现场造一个技能：report-writer（含 references/，用来演示渐进式披露）。"""
    skill_dir = SKILLS_ROOT / "report-writer"
    (skill_dir / "references").mkdir(parents=True, exist_ok=True)

    # ---- SKILL.md：唯一必需的文件，元数据 + 指令 ----
    (skill_dir / "SKILL.md").write_text(
        """---
name: report-writer
description: 撰写正式工作周报时使用本技能。用户提到周报、日报、工作总结时加载。
---

# 周报撰写规范

1. 结构固定为三段：本周完成 / 下周计划 / 风险与求助
2. 每条用「动宾短语」开头，例如「完成了 XX 模块开发」
3. 总字数控制在 200 字以内

## 更多要求

如果用户没有特别说明，默认面向技术团队负责人汇报。
详细的行文口吻与用词禁忌见 `references/style.md`（需要时再读）。
""",
        encoding="utf-8",
    )

    # ---- references/：渐进式披露的「第二层」，只有真的用到才会被 read_file ----
    (skill_dir / "references" / "style.md").write_text(
        """# 行文风格

- 不写「我觉得」「大概」「可能」，一律给确定结论
- 不写与本周无关的背景铺垫
- 风险项必须写清「影响面 + 需要的支持」
""",
        encoding="utf-8",
    )
    return skill_dir

In [ ]:
def part1_local_skills() -> None:
    """Part 1：技能放在真实磁盘上（对应课案的 deepagents skills 代码）。"""
    print("=" * 66)
    print("Part 1：本地技能目录 + LocalShellBackend（课案的 deepagents skills 写法）")
    print("=" * 66)

    skill_dir = build_local_skill()
    print(f"技能父目录（skills 参数传的就是它）：{SKILLS_ROOT}")
    print(f"技能目录结构：")
    for path in sorted(SKILLS_ROOT.rglob("*")):
        if path.is_file():
            print(f"    {path.relative_to(SKILLS_ROOT)}")
    print()

    agent = create_deep_agent(
        model=llm,
        # 课案用 LocalShellBackend，是因为它的技能里有转换脚本要执行；
        # 这里保持同一类后端，同时也能执行技能里可能带的 scripts/。
        backend=LocalShellBackend(
            root_dir=str(SKILLS_ROOT),
            inherit_env=True,
            env={"PATH": VENV_SCRIPTS + os.pathsep + os.environ.get("PATH", "")},
        ),
        # 课案是 skills=["/"]（扫描根目录）；这里指向技能的父目录，语义相同但更安全。
        # 关键：传父目录，SkillsMiddleware 会自己去里面找 */SKILL.md。
        skills=[str(SKILLS_ROOT)],
        system_prompt="你是职场写作助手。需要写周报时，先读技能文件按规范来写。",
    )

    # 用 updates 模式把工具调用打出来 —— 这是「渐进式披露」最直接的证据：
    # 启动时系统提示里只有 description，模型决定用技能之后才会 read_file 拉 SKILL.md 正文。
    print("执行任务（同时打印 Agent 的工具调用轨迹）：")
    for chunk in agent.stream(
        {"messages": [("user", "帮我写本周周报：完成了登录模块，下周做支付")]},
        stream_mode="updates",
        # 技能任务链路更长（选技能 → 读 SKILL.md → 可能再读 references → 写正文），放宽步数
        config={"recursion_limit": 50},
    ):
        for node_name, update in chunk.items():
            # 有的中间件钩子节点只写非消息字段，取不到 messages 就当空列表处理
            messages = update.get("messages", []) if isinstance(update, dict) else []
            for message in messages:
                # ① 模型这一轮决定调用哪些工具、传什么参数（read_file / write_file 都在这里现形）
                for call in getattr(message, "tool_calls", None) or []:
                    print(f"  [{node_name}] → {call.get('name')}({call.get('args')})")
                # ② 工具返回：只打印长度，避免把整个 SKILL.md 正文刷到屏幕上
                if type(message).__name__ == "ToolMessage":
                    print(f"  [{node_name}] ← {message.name} 返回 {len(str(message.content))} 字符")
                # ③ 其余带正文的消息 = 模型写出来的周报本身
                elif getattr(message, "content", ""):
                    print(f"\n【周报】\n{message.content}\n")

    print("注意上面是否出现 read_file 读取 SKILL.md —— 那就是「匹配到才加载」。")

In [ ]:
def part2_cloud_skills() -> None:
    """Part 2：技能存进 Store，按用户 namespace 隔离（对应课案的「云技能」小节）。"""
    print("\n" + "=" * 66)
    print("Part 2：云技能 —— skills 指向 Store 里的 /skills/（课案「云技能」小节）")
    print("=" * 66)

    if not settings.pg_uri:
        print("[跳过] 未配置 settings.pg_uri（.env 里的 PG_URI），无法演示云端技能仓库。")
        return

    with (
        PostgresStore.from_conn_string(settings.pg_uri) as store,
        PostgresSaver.from_conn_string(settings.pg_uri) as checkpointer,
    ):
        store.setup()
        checkpointer.setup()

        # 为不同用户预存不同的 skill：这里演示 user-001 的两个技能。
        # key 的写法有讲究："/skills/code-review/SKILL.md" 是课案原样路径（会静默失效），
        # "/code-review/SKILL.md" 是剥掉前缀后的正确路径（才会被 SkillsMiddleware 找到）。
        store.put(
            ("user-001",),
            "/skills/code-review/SKILL.md",
            create_file_data(
                """---
name: code-review
description: Python 代码审查，检查规范性和性能问题
---

你是 Python 专家，审查以下代码的规范性、性能和安全性。
输出格式：先列问题（按严重程度排序），再给修改后的代码。
"""
            ),
        )
        # 同上，另存一份「剥掉 /skills/ 前缀」的 key —— 这一份才会被 SkillsMiddleware 找到
        store.put(
            ("user-001",),
            "/code-review/SKILL.md",
            create_file_data(
                """---
name: code-review
description: Python 代码审查，检查规范性和性能问题
---

你是 Python 专家，审查以下代码的规范性、性能和安全性。
输出格式：先列问题（按严重程度排序），再给修改后的代码。
"""
            ),
        )
        store.put(
            ("user-001",),
            "/skills/sql-gen/SKILL.md",
            create_file_data(
                """---
name: sql-gen
description: 根据自然语言生成 SQL 语句
---

根据用户的自然语言描述生成对应的 SQL 语句。
要求：只输出一段 SQL，表名用 users / orders，不要解释。
"""
            ),
        )
        # 同上：sql-gen 也补一份正确 key
        store.put(
            ("user-001",),
            "/sql-gen/SKILL.md",
            create_file_data(
                """---
name: sql-gen
description: 根据自然语言生成 SQL 语句
---

根据用户的自然语言描述生成对应的 SQL 语句。
要求：只输出一段 SQL，表名用 users / orders，不要解释。
"""
            ),
        )
        print("已向 Store 的 namespace ('user-001',) 预存 2 个技能：code-review / sql-gen")
        print("（Store 里的 key 同时写了课案原样路径与「剥掉 /skills/ 前缀」的正确路径）\n")

        agent = create_deep_agent(
            model=llm,
            # 技能仓库在 Store 里，路径前缀 /skills/
            skills=["/skills/"],
            backend=CompositeBackend(
                default=StateBackend(),
                routes={
                    "/skills/": StoreBackend(
                        # 本地用固定值；生产改成 lambda rt: (rt.server_info.user.identity,)
                        # 就能做到「每个用户看到自己的技能集」
                        namespace=lambda rt: ("user-001",),
                    ),
                },
            ),
            store=store,
            checkpointer=checkpointer,
        )

        # thread_id 用带前缀的名字，避免和同目录其他文件（它们也用 PostgresSaver
        # 往同一个库写检查点，课案里写的是 "1"）串到同一条对话历史上去。
        config = {"configurable": {"thread_id": "jxsd-skills-1"}}
        result = agent.invoke(
            {"messages": [{"role": "user", "content": "用技能sql-gen查询用户总数"}]},
            config={**config, "recursion_limit": 50},
        )
        # 和 Part 1 一样打印**完整消息轨迹**，方便确认技能有没有被真的加载：
        # 只要看到 read_file("/skills/sql-gen/SKILL.md")，就说明云技能被发现了。
        for m in result["messages"]:
            print(m)
            print("-" * 50)

        print("\n最终回答：")
        print(result["messages"][-1].content)
        print(
            "\n要点：技能文件不在磁盘上，而是存在 Store 的 ('user-001',) 命名空间里；\n"
            "      换一个 namespace 就是另一套技能 —— 这就是课案说的「云端多用户隔离」。"
        )


part1_local_skills()
part2_cloud_skills()

### 预期输出

```text
Part 1：本地技能目录 + LocalShellBackend（课案的 deepagents skills 写法）
技能父目录（skills 参数传的就是它）：…\tmp_nb_work\deepagents_skills
技能目录结构：
    report-writer\references\style.md
    report-writer\SKILL.md

执行任务（同时打印 Agent 的工具调用轨迹）：
  [model] → read_file({'file_path': '…/report-writer/SKILL.md', ...})
  [tools] ← read_file 返回 NNN 字符
【周报】
（按技能写出的三段式周报，措辞每次不同）

Part 2：云技能 —— skills 指向 Store 里的 /skills/
已向 Store 的 namespace ('user-001',) 预存 2 个技能：code-review / sql-gen
...
最终回答：
（生成的 SQL，措辞每次不同）
```

> ⚠️ 模型是否真的 `read_file` 拉 SKILL.md、周报/SQL 正文**都由模型决定、每次不同**；
> 技能父目录的绝对路径**因机器而异**。稳定的是「技能目录结构」两行、「预存 2 个技能」
> 那两行，以及 Part 2 末尾「要点：…云端多用户隔离」那段固定文案。

## 5. 自定义 Backend（官方文档补充）

课案讲了七种内置后端怎么用，**没讲怎么造一个自己的后端**。这一节对照官方文档
backends.mdx 的「Custom backends」与「Add policy hooks」补上这块。

为什么不直接用内置的七个？真实项目常遇到这些需求，只能自己写后端：

- 文件根本不在磁盘上，而在**数据库 / 对象存储 / 内部配置中心**；
- 每次读写都要**审计**或**内容校验**（禁止写入密钥）；
- 需要**限流**（后端级 QPS 上限）或**只读模式**。

写自定义后端前必须知道的两个关键事实（本地实测 deepagents 0.7.13）：

- `BackendProtocol` **没有任何强制抽象方法** —— 18 个方法（9 同步 + 9 异步）全带默认实现，
  想支持什么就重写什么；**没重写的方法会在调用时抛 `NotImplementedError` 并打断整个运行**；
- 所有操作都返回**结构化结果对象**（dataclass），失败靠 `error` 字段表达、**不抛异常**。

这一节**全离线、0 次真实模型调用**，用一个「剧本模型」`ScriptedModel` 依次吐预设的
AIMessage 驱动 Agent，输出完全确定、可复现。

In [ ]:
import time

from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_openai import ChatOpenAI
from pydantic import PrivateAttr

from deepagents import create_deep_agent
from deepagents.backends import BackendProtocol
from deepagents.backends.protocol import (
    DeleteResult,
    EditResult,
    LsResult,
    ReadResult,
    WriteResult,
)
# 官方提供的两个小工具（自己手写分页逻辑极易和框架契约不一致，直接用它们最稳）：
#   create_file_data  → 把字符串包装成后端内部的 FileData 结构（dict）
#   slice_read_response → 按 offset/limit 切片并生成合法的 ReadResult（含 1 起算行号）
from deepagents.backends.utils import create_file_data, file_data_to_string, slice_read_response


# ================================================================
# 剧本模型（与 14_上下文治理_官方补充.py 同一手法）
# ================================================================
def ai_tool_call(name: str, args: dict, call_id: str) -> AIMessage:
    """构造一条「模型要调工具」的 AIMessage。"""
    return AIMessage(
        content="",
        tool_calls=[{"name": name, "args": args, "id": call_id, "type": "tool_call"}],
    )


class ScriptedModel(ChatOpenAI):
    """按剧本依次吐消息的假模型：只覆写 _generate，其余框架方法沿用真实现。"""

    _script: list = PrivateAttr(default_factory=list)
    _cursor: int = PrivateAttr(default=0)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        message = self._script[self._cursor]
        self._cursor += 1
        return ChatResult(generations=[ChatGeneration(message=message)])


def make_scripted(script: list) -> ScriptedModel:
    model = ScriptedModel(model="scripted", api_key="offline", base_url="http://localhost:9")
    model._script = script
    return model


def show_tool_messages(result: dict, limit: int = 100) -> None:
    for message in result.get("messages", []):
        if message.type == "tool":
            body = str(message.content).replace("\n", " ")
            print(f"    [ToolMessage] {getattr(message, 'name', '?')}: {body[:limit]}")

In [ ]:
# ================================================================
# Demo 1：最小自定义后端 —— 把「文件」放在内存字典里
# ================================================================
# 自定义后端的套路（官方 backends.mdx）：
#     1. 继承 BackendProtocol；
#     2. 重写你支持的操作，返回对应的 XxxResult 对象；
#     3. 失败时 **返回带 error 的结果**（别抛异常）；
#     4. create_deep_agent(backend=你的实例) 挂上去，文件工具立刻指向它。
class DictBackend(BackendProtocol):
    """把文件存在内存字典里的最小后端：只实现 write / read / ls 三个方法。

    这正是官方 backends.mdx 说的用法 ——「想支持什么就重写什么」：
    代理能写、能读、能列目录；glob / grep / edit 等没实现的走父类默认实现。
    （真实项目里把 dict 换成数据库或对象存储的读写即可，方法签名不用变。）
    """

    def __init__(self) -> None:
        # 结构：{"/notes/a.txt": FileData} —— 注意存的是框架的 FileData（dict），
        # 不是裸字符串：用官方 create_file_data() 包一下，读的时候才能交给
        # slice_read_response() 正确分页。
        self.files: dict[str, dict] = {}
        # 审计日志：自定义后端的附加价值之一（内置后端不会帮你记这个）
        self.audit: list[str] = []

    # ---------- 写 ----------
    def write(self, file_path: str, content: str) -> WriteResult:
        self.files[file_path] = create_file_data(content)
        self.audit.append(f"WRITE {file_path} ({len(content)} 字符)")
        return WriteResult(path=file_path, error=None)

    # ---------- 读（分页与行号契约交给官方 helper，别自己算）----------
    def read(self, file_path: str, offset: int = 0, limit: int = 2000) -> ReadResult:
        if file_path not in self.files:
            # 失败不抛异常，而是把原因塞进 error 字段
            return ReadResult(error=f"文件不存在：{file_path}")
        # slice_read_response 内部会做边界钳制、生成 1 起算的 start_line/end_line，
        # 并在越界时返回带 error 的结果（自己手写会踩 "1 <= start_line" 的校验）
        return slice_read_response(self.files[file_path], offset, limit)

    # ---------- 列目录（非递归，条目结构照官方 FileInfo：path / is_dir / size / modified_at）----------
    def ls(self, path: str) -> LsResult:
        normalized = path if path.endswith("/") else path + "/"
        entries: list[dict] = []
        subdirs: set[str] = set()
        for file_path, file_data in sorted(self.files.items()):
            if not file_path.startswith(normalized):
                continue
            relative = file_path[len(normalized):]
            if "/" in relative:
                # 属于更深层的文件 → 只把「直接子目录」汇总成一条，不递归展开
                subdirs.add(normalized + relative.split("/")[0] + "/")
                continue
            entries.append({
                "path": file_path,
                "is_dir": False,
                # 官方 FileInfo.size 是**字节数**，不是字符数（中文一字 3 字节）
                "size": len(file_data_to_string(file_data).encode("utf-8")),
                "modified_at": file_data.get("modified_at", "") if isinstance(file_data, dict) else "",
            })
        entries.extend({"path": d, "is_dir": True, "size": 0, "modified_at": ""} for d in sorted(subdirs))
        return LsResult(entries=entries)

In [ ]:
def demo_1_minimal_backend() -> None:
    print("=" * 70)
    print("Demo 1：最小自定义后端 —— 只实现 write / read / ls")
    print("=" * 70)

    backend = DictBackend()
    # 先手工塞两个文件，方便后面 ls 有东西看
    backend.write("/notes/todo.txt", "买牛奶\n写周报")
    backend.write("/notes/idea.txt", "用自定义后端接内部知识库")

    # ---- Part A：只用我们实现了的三个能力 ----
    agent = create_deep_agent(
        model=make_scripted([
            ai_tool_call("write_file", {"file_path": "/notes/new.txt", "content": "代理写入的内容"}, "c1"),
            ai_tool_call("ls", {"path": "/notes"}, "c2"),
            ai_tool_call("read_file", {"file_path": "/notes/new.txt"}, "c3"),
            AIMessage(content="自定义后端读写都正常。"),
        ]),
        backend=backend,     # ← 关键一行：把文件工具指向我们自己的后端
    )
    result = agent.invoke({"messages": [{"role": "user", "content": "写个文件然后看看目录"}]})
    show_tool_messages(result, limit=120)
    print(f"\n  后端里的文件：{sorted(backend.files)}")
    print("  后端自己记的审计日志：")
    for line in backend.audit:
        print(f"    {line}")
    print(
        "  ↑ 代理根本没碰磁盘 —— write_file / ls / read_file 全部落在我们自己的 dict 上。\n"
        "    真实项目把 dict 换成数据库或对象存储，就是一个「数据库后端」。\n"
        "    注意 audit 这份日志：内置后端不会给你记，这是自定义后端最容易加的价值。"
    )

    # ---- Part B：没实现的方法会怎样？（实测，很重要）----
    print("\n  --- Part B：调用一个我们**没实现**的能力（grep）---")
    probe_backend = DictBackend()
    probe_backend.write("/notes/a.txt", "里面有知识库三个字")
    probe_agent = create_deep_agent(
        model=make_scripted([
            ai_tool_call("grep", {"pattern": "知识库"}, "c1"),
            AIMessage(content="（不该走到这里）"),
        ]),
        backend=probe_backend,
    )
    try:
        probe_agent.invoke({"messages": [{"role": "user", "content": "搜一下知识库"}]})
    except Exception as exc:  # noqa: BLE001
        print(f"    抛错中断：{type(exc).__name__}: {str(exc)[:90]}")
        print(
            "    ↑ 结论（本机实测路径）：**没重写的方法走父类默认实现，调用时直接抛错、把运行打断** ——\n"
            "      因为 ToolNode 默认只把 ToolInvocationError 转成错误消息，其余异常一律 re-raise。\n"
            "      所以两种做法二选一：① 干脆实现它；② 显式重写成「返回带 error 的结果」，\n"
            "      让模型看到一句说明而不是让运行崩掉（Demo 2 的只读后端就是 ② 的写法）。"
        )
    else:
        print(
            "    ↑ 本次调用**成功**了 —— 说明这个版本的父类给该方法补了可用的默认实现。\n"
            "      这也提醒我们：**别按印象断言「某个方法一定没有默认实现」**，以运行结果为准；\n"
            "      但返回值是否可靠仍要看文档（Demo 2 的只读后端是更稳的写法：显式返回 error）。"
        )

In [ ]:
# ================================================================
# Demo 2：只读后端 —— 给代理一个「只能看不能改」的知识库
# ================================================================
# 只读是最常见的自定义需求。做法有两种：
#     a) 重写 write/edit/delete，返回 error（工具还在，但一写就被拒）；
#     b) 用权限规则 FilesystemPermission(mode="deny")（14_上下文治理_官方补充.py Demo 2）——
#        那是「不改后端」的做法，适合权限按调用方/路径动态变化的场景。
# 本 Demo 用 a)，顺便看看「写被拒」时模型收到的是什么。
class ReadOnlyBackend(DictBackend):
    """在 DictBackend 基础上禁掉所有写操作：适合把内部文档库暴露给代理。"""

    _REJECT = "该知识库为只读模式，不允许修改"

    def write(self, file_path: str, content: str) -> WriteResult:
        return WriteResult(error=self._REJECT, path=file_path)

    def edit(self, file_path: str, old_string: str, new_string: str, replace_all: bool = False) -> EditResult:
        return EditResult(error=self._REJECT, path=file_path)

    def delete(self, file_path: str) -> DeleteResult:
        return DeleteResult(error=self._REJECT, path=file_path)


def demo_2_readonly_backend() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：只读后端 —— 写操作被拒（返回 error 而不是抛异常）")
    print("=" * 70)

    backend = ReadOnlyBackend()
    # 注意：只读后端不能靠 write() 塞数据（会被自己拒掉），要直接往存储里放 ——
    # 但**必须用 create_file_data 包装**，存裸字符串会让后面的读取报
    # TypeError: string indices must be integers（实测踩过）
    backend.files["/wiki/onboarding.md"] = create_file_data("新员工入职指南：第一天领电脑……")

    agent = create_deep_agent(
        model=make_scripted([
            ai_tool_call("read_file", {"file_path": "/wiki/onboarding.md"}, "c1"),
            ai_tool_call("write_file", {"file_path": "/wiki/hack.md", "content": "改一下指南"}, "c2"),
            AIMessage(content="文档读到了，写入被拒（只读库）。"),
        ]),
        backend=backend,
    )
    result = agent.invoke({"messages": [{"role": "user", "content": "看看入职指南，顺手改一下"}]})
    show_tool_messages(result, limit=110)
    print(f"\n  后端里的文件（应当没有 hack.md）：{sorted(backend.files)}")
    print(
        "  ↑ 写被拒时，模型收到的是**一句说明**（ToolMessage），运行没有崩 ——\n"
        "    这正是「结构化结果代替异常」的用处：拒绝是业务语义，不是程序错误。"
    )

In [ ]:
# ================================================================
# Demo 3：把「审计 + 内容校验」做成策略钩子
# ================================================================
# 官方 backends.mdx 的「Add policy hooks」讲的是在后端层统一做限流/审计/校验 ——
# 相比在每个工具函数里各写一遍，放在后端层的好处是**所有入口都绕不过去**
# （模型的 write_file、edit_file、子代理的写入，最终都走后端的方法）。
class PolicyBackend(DictBackend):
    """带内容校验与写入限流的后端。"""

    def __init__(self, max_writes_per_second: int = 2) -> None:
        super().__init__()
        self.max_writes_per_second = max_writes_per_second
        self._write_times: list[float] = []
        self.rejected: list[str] = []

    def write(self, file_path: str, content: str) -> WriteResult:
        # ---- 策略 1：内容校验（禁止把密钥写进文件）----
        import re

        if re.search(r"sk-[A-Za-z0-9]{8,}", content):
            self.rejected.append(f"内容校验拦截 {file_path}")
            return WriteResult(error="检测到疑似 API 密钥，已拒绝写入", path=file_path)

        # ---- 策略 2：写入限流（滑动窗口）----
        now = time.time()
        self._write_times = [t for t in self._write_times if now - t < 1.0]
        if len(self._write_times) >= self.max_writes_per_second:
            self.rejected.append(f"限流拦截 {file_path}")
            return WriteResult(error="写入过于频繁，请稍后再试", path=file_path)
        self._write_times.append(now)

        result = super().write(file_path, content)
        self.audit.append(f"AUDIT {file_path} 写入通过（内容校验 + 限流都过了）")
        return result


def demo_3_policy_hooks() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：策略钩子 —— 内容校验 + 写入限流（都在后端层统一做）")
    print("=" * 70)

    backend = PolicyBackend(max_writes_per_second=2)
    agent = create_deep_agent(
        model=make_scripted([
            ai_tool_call("write_file", {"file_path": "/work/a.txt", "content": "正常内容 A"}, "c1"),
            ai_tool_call("write_file", {"file_path": "/work/b.txt", "content": "正常内容 B"}, "c2"),
            ai_tool_call("write_file", {"file_path": "/work/c.txt", "content": "正常内容 C"}, "c3"),
            # 故意写一条"疑似密钥"的内容来触发策略拦截。
            # 这里用一眼可辨的假 key（真实场景里是用户不小心把真 key 写进文件）；
            # 用 FAKE 字样的另一个好处：不会被本仓库的密钥扫描脚本误报。
            ai_tool_call("write_file", {"file_path": "/work/secret.txt",
                                        "content": "sk-FAKEKEYFORDEMO1234567890"}, "c4"),
            AIMessage(content="有几次写入被策略拦下了。"),
        ]),
        backend=backend,
    )
    result = agent.invoke({"messages": [{"role": "user", "content": "写几个文件"}]})
    show_tool_messages(result, limit=80)
    print(f"\n  实际写入成功的文件：{sorted(backend.files)}")
    print(f"  被策略拒绝的记录：{backend.rejected}")
    # 断言兜住：三次写调用必须在 1 秒滑窗内完成，否则限流那条结论就不成立
    # （脚本模型下必然成立；万一机器慢到这个阈值失真，这里会直接报出来而不是印错结论）
    assert len(backend.rejected) == 2, f"预期两类策略各拦一次，实际：{backend.rejected}"
    print(f"  审计日志（通过策略的那几次）：{backend.audit}")
    print(
        "  ↑ 三次正常写入里，第 3 次被**限流**拦下（1 秒内最多 2 次）；\n"
        "    带密钥的那次被**内容校验**拦下。两类策略都写在后端层 ——\n"
        "    无论模型怎么绕（换工具、换子代理），只要落地到文件系统就得过这一关。"
    )


demo_1_minimal_backend()
demo_2_readonly_backend()
demo_3_policy_hooks()
print("\n全部 Demo 执行完毕（0 次真实模型调用，离线可复现）。")

### 预期输出

```text
Demo 1：最小自定义后端 —— 只实现 write / read / ls
    [ToolMessage] write_file: Updated file /notes/new.txt
    [ToolMessage] ls: ['/notes/idea.txt', '/notes/new.txt', '/notes/todo.txt']
    [ToolMessage] read_file: 1  代理写入的内容
  后端里的文件：['/notes/idea.txt', '/notes/new.txt', '/notes/todo.txt']
  后端自己记的审计日志：
    WRITE /notes/todo.txt (7 字符)
    WRITE /notes/idea.txt (12 字符)
    WRITE /notes/new.txt (7 字符)
  --- Part B：调用一个我们**没实现**的能力（grep）---
    抛错中断：NotImplementedError:

Demo 2：只读后端 —— 写操作被拒（返回 error 而不是抛异常）
    [ToolMessage] read_file: 1  新员工入职指南：第一天领电脑……
    [ToolMessage] write_file: 该知识库为只读模式，不允许修改
  后端里的文件（应当没有 hack.md）：['/wiki/onboarding.md']

Demo 3：策略钩子 —— 内容校验 + 写入限流（都在后端层统一做）
    [ToolMessage] write_file: Updated file /work/a.txt
    [ToolMessage] write_file: Updated file /work/b.txt
    [ToolMessage] write_file: 写入过于频繁，请稍后再试
    [ToolMessage] write_file: 检测到疑似 API 密钥，已拒绝写入
  实际写入成功的文件：['/work/a.txt', '/work/b.txt']
  被策略拒绝的记录：['限流拦截 /work/c.txt', '内容校验拦截 /work/secret.txt']
  审计日志（通过策略的那几次）：['WRITE /work/a.txt (6 字符)', 'AUDIT /work/a.txt 写入通过（内容校验 + 限流都过了）', 'WRITE /work/b.txt (6 字符)', 'AUDIT /work/b.txt 写入通过（内容校验 + 限流都过了）']

全部 Demo 执行完毕（0 次真实模型调用，离线可复现）。
```

这一节用剧本模型驱动，输出**完全确定、可复现**，上面就是本机实测抓下来的真实输出。

## 小结

- **人工审核**：`interrupt_on` + `checkpointer` 让危险工具执行前挂起；决策用
  `Command(resume={"decisions": [...]})` 送回冻结的图，恢复必须用同一个 thread_id；
- **记忆**：记忆就是文件，`memory=[...]` + `store=` 实现跨会话；`CompositeBackend`
  只把 `/memories/` 路由到 StoreBackend；**预存文件的 Store key 要写成「剥掉路由前缀」后的路径**；
- **子智能体**：`subagents=[SubAgent(...)]` 让主 Agent 通过 `task` 工具委派；
  `name` 必须唯一（重名直接 `ValueError`），`model` 可选（不传就继承主 Agent）；
- **Skills**：一个文件夹 + 一个 `SKILL.md`，三步渐进式披露省 Token；`skills` 传**父目录**；
  技能仓库可以是磁盘也可以是 Store（云技能靠 namespace 隔离）；
- **自定义后端**：继承 `BackendProtocol`，只重写需要的方法，失败返回带 `error` 的结果
  （不抛异常）；没重写的方法调用时会 `NotImplementedError` 打断运行，要么实现它、
  要么显式重写成「返回 error」。

## 常见坑

1. **`interrupt_on` 不配 checkpointer 直接报错** —— 中断状态要有地方存。
2. **中断恢复必须用同一个 thread_id**，否则接不上原来的执行线。
3. **`input()` 在无头执行时必崩** —— 要么 `isatty()` 判断，要么 `scripted_input` 替换
   `builtins.input`（预设答案用完要兜底，否则 `IndexError`）。
4. **三个 SubAgent 重名会 `ValueError: Duplicate subagent name`** —— 复制粘贴改名字。
5. **`SubAgent.model` 是可选字段**，会被 `spec.get("model", model)` 兜底成主 Agent 模型，
   不是必填。
6. **经过 CompositeBackend 路由时，调用方路径会被剥掉前缀** —— 预存记忆/技能的 Store key
   要写成「剥掉前缀后」的路径，否则 `File not found` / 技能发现数 = 0（静默失效）。
7. **`read_file` 返回带行号**，模型会把带行号的整行当 `old_string` 传给 `edit_file` 导致
   `String not found` —— 在 system_prompt 里把格式说清楚，并给 write_file 兜底。
8. **自定义后端不要抛异常** —— 抛了打断整个运行；返回 error 结果只是让模型看到一句拒绝说明。
9. **`skills` 传的是父目录，不是 SKILL.md 本身**；传根目录 `/` 会把整台机器当技能仓库翻。

## 官方链接

- 定制（customization，interrupt_on / memory / subagents / skills 汇总）：<https://docs.langchain.com/oss/python/deepagents/customization>
- 子智能体（subagents）：<https://docs.langchain.com/oss/python/deepagents/subagents>
- 技能（skills）：<https://docs.langchain.com/oss/python/deepagents/skills>
- 后端（backends，含 Custom backends / policy hooks）：<https://docs.langchain.com/oss/python/deepagents/backends>